In [1]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from pandas import DataFrame, concat
from pickle import load
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/potency_min.pickle', 'rb') as file:
    potency = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [3]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

In [4]:
X = descriptors_transformer.transform(molecules)
Y = potency['Potency']
XY = concat((X, Y), axis=1)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20)

In [6]:
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [7]:
scaler_y = StandardScaler().fit(DataFrame(y_train))
y_train_scal = scaler_y.transform(DataFrame(y_train))
y_test_scal = scaler_y.transform(DataFrame(y_test))

In [ ]:
gb = HistGradientBoostingRegressor()
pg = {'learning_rate': [0.001, 0.01, 0.1, 1],
      'max_depth': [None, 1, 3, 5, 7, 10],
      'max_iter': [50, 100, 200, 300],
      'max_leaf_nodes': [10, 20, 30, 40, 50],
      'min_samples_leaf': [5, 10, 20, 30],
      'l2_regularization': [0, 0.3, 0.6, 0.9]
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(gb, pg, verbose=3, cv=cv)

gs.fit(x_train_scal, y_train_scal.ravel())

Fitting 25 folds for each of 36 candidates, totalling 900 fits
[CV 1/25] END ..max_depth=None, n_estimators=50;, score=0.144 total time=   2.5s
[CV 2/25] END ..max_depth=None, n_estimators=50;, score=0.182 total time=   2.3s
[CV 3/25] END ..max_depth=None, n_estimators=50;, score=0.140 total time=   2.3s
[CV 4/25] END ..max_depth=None, n_estimators=50;, score=0.125 total time=   2.3s
[CV 5/25] END ..max_depth=None, n_estimators=50;, score=0.169 total time=   2.3s
[CV 6/25] END ..max_depth=None, n_estimators=50;, score=0.168 total time=   2.3s
[CV 7/25] END ..max_depth=None, n_estimators=50;, score=0.139 total time=   2.3s
[CV 8/25] END ..max_depth=None, n_estimators=50;, score=0.157 total time=   2.3s
[CV 9/25] END ..max_depth=None, n_estimators=50;, score=0.136 total time=   2.3s
[CV 10/25] END .max_depth=None, n_estimators=50;, score=0.132 total time=   2.3s
[CV 11/25] END .max_depth=None, n_estimators=50;, score=0.129 total time=   2.3s
[CV 12/25] END .max_depth=None, n_estimators=5

In [1]:
gs.best_estimator_

NameError: name 'gs' is not defined

In [10]:
gs = HistGradientBoostingRegressor()
gs.fit(x_train_scal, y_train_scal.ravel())

HistGradientBoostingRegressor()

In [11]:
y_pred = gs.predict(x_test_scal)
print(f'R^2 = {r2_score(y_test_scal, y_pred)}')
print(f'RMSE = {root_mean_squared_error(y_test_scal, y_pred)}')

R^2 = 0.15024658264252788
RMSE = 0.9242733810353972


In [30]:
DataFrame(y_test).value_counts()

Potency  
89.12510     707
50.11870     683
79.43280     379
47.39355     341
44.66840     319
            ... 
3.91515        1
2.99035        1
149.87150      1
1.99530        1
177.82800      1
Name: count, Length: 398, dtype: int64